# Auto-encoding variable bayes

Adapted from https://github.com/pytorch/examples/tree/main/vae

In [ ]:
from __future__ import print_function
import argparse
import torch
import torch.utils.data
from torch import nn, optim

from torch.nn import functional as F
from torch.optim import Optimizer
from torchvision import datasets, transforms
from torchvision.utils import save_image

import torchastic
import matplotlib.pyplot as plt

from typing import Union


In [ ]:

batch_size: int = 128
"""batch size for training (default: 128)"""

epochs: int = 10
"""number of epochs to train (default: 10)"""
_use_accel: bool = True
"""should accelerator be enabled"""
seed: int = 1
"""rng seed (default: 1)"""
log_interval: int = 10
"""how many batches to wait before logging training status (default: 10)"""

use_accel: bool = _use_accel and torch.accelerator.is_available()

torch.manual_seed(seed)

if use_accel:
    device = torch.accelerator.current_accelerator()
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

In [ ]:
kwargs = {'num_workers': 1, 'pin_memory': True} if use_accel else {}
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST('../data', train=True, download=True,
                   transform=transforms.ToTensor()),
    batch_size=batch_size, shuffle=True, **kwargs)
test_loader = torch.utils.data.DataLoader(
    datasets.MNIST('../data', train=False, transform=transforms.ToTensor()),
    batch_size=batch_size, shuffle=False, **kwargs)

In [ ]:
class VAE(nn.Module):
    """
    An auto-encoding model using a variable bayesian approach.
    This version uses
    """
    def __init__(self, name: str):
        super(VAE, self).__init__()
        self.name: str = name
        self.fc1 = nn.Linear(784, 400)
        self.fc21 = nn.Linear(400, 20)
        self.fc22 = nn.Linear(400, 20)
        self.fc3 = nn.Linear(20, 400)
        self.fc4 = nn.Linear(400, 784)


    def encode(self, x):
        h1 = F.relu(self.fc1(x))
        return self.fc21(h1), self.fc22(h1)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std

    def decode(self, z):
        h3 = F.relu(self.fc3(z))
        return torch.sigmoid(self.fc4(h3))

    def forward(self, x):
        mu, logvar = self.encode(x.view(-1, 784))
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

    def get_name(self) -> str:
        return self.name

    def get_device(self) -> torch.device:
        return next(self.parameters()).device

    def get_dtype(self) -> torch.dtype:
        return next(self.parameters()).dtype


In [ ]:
model_32: VAE = VAE("adam_32").to(device=device, dtype=torch.float32)
optimizer_32: optim.Optimizer = optim.Adam(model_32.parameters(), lr=1e-3)

model_w_32: VAE = VAE("adamW_32").to(device=device, dtype=torch.float32)
optimizer_w_32: optim.Optimizer = optim.AdamW(model_w_32.parameters(), lr=1e-3)

model_w_16: VAE = VAE("adamW_16").to(device=device, dtype=torch.bfloat16)
optimizer_w_16: optim.Optimizer = optim.AdamW(model_w_16.parameters(), lr=1e-3)

model_sr_16: VAE = VAE("sr_adamW_16").to(device=device, dtype=torch.bfloat16)
optimizer_sr_16: optim.Optimizer = torchastic.AdamW(model_sr_16.parameters(), lr=1e-3)

In [ ]:
# Reconstruction + KL divergence losses summed over all elements and batch
def loss_function(recon_x, x, mu, logvar,
                  dtype: torch.dtype = torch.get_default_dtype()
    ):
    BCE = F.binary_cross_entropy(recon_x, x.view(-1, 784), reduction='sum')

    # see Appendix B from VAE paper:
    # Kingma and Welling. Auto-Encoding Variational Bayes. ICLR, 2014
    # https://arxiv.org/abs/1312.6114
    # 0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dtype=dtype)

    return torch.sum(BCE + KLD, dtype=dtype) #BCE + KLD


In [ ]:
def train(epoch, model: VAE, optimizer: Optimizer):
    model.train()
    train_loss = 0
    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device=device, dtype=model.get_dtype())
        optimizer.zero_grad()
        recon_batch, mu, logvar = model(data)
        loss = loss_function(recon_batch, data, mu, logvar, dtype=model.get_dtype())
        loss.backward()
        train_loss += loss.item()
        optimizer.step()
        if batch_idx % log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader),
                loss.item() / len(data)))

    print('====> Epoch: {} Average loss: {:.4f}'.format(
          epoch, train_loss / len(train_loader.dataset)))


In [ ]:

def test(epoch, model: VAE, optimizer: Optimizer) -> tuple[torch.Tensor, Union[None, str], str] :
    model.eval()
    test_loss = 0
    comp_img: Union[None, str] = None
    comp_epoch: str = ""
    with torch.no_grad():
        for i, (data, _) in enumerate(test_loader):
            data = data.to(device).to(device=device, dtype=model.get_dtype())
            recon_batch, mu, logvar = model(data)
            test_loss += loss_function(recon_batch, data, mu, logvar).item()
            if i == 0:
                n = min(data.size(0), 8)
                comparison = torch.cat([data[:n],
                                      recon_batch.view(batch_size, 1, 28, 28)[:n]])
                comp_img = 'results/reconstruction_' + str(epoch) + '_' + str(model.get_name()) +  '.png'
                save_image(comparison.cpu(),
                         comp_img, nrow=n)
                #comp_img = comparison.cpu().permute(0,2,3,1).reshape(2 * 28, 8 * 28, 1)
                comp_epoch = str(epoch)

    test_loss /= len(test_loader.dataset)
    print('====> Test set loss: {:.4f}  {:s}'.format(test_loss, model.get_name()))
    return test_loss, comp_img, comp_epoch

In [ ]:

def test_and_train(model: VAE, optimizer: Optimizer) -> None:
    images: list[torch.Tensor] = []
    captions: list[str] = []
    #fig_scale : float = 2
    #plt.figure(figsize=(0.5+(2*fig_scale),0.25+(epochs * fig_scale)))
    #plt.title(f"Epoch outputs for model {str(model.get_name())}")

    for epoch in range(1, epochs + 1):
        train(epoch, model, optimizer)
        loss, comp_img, comp_epoch = test(epoch, model, optimizer)

        if comp_img is not None:
            pass
            #plt.subplot(epochs, 2 ,(2*epoch)-1)
            #plt.imshow(comp_img.to(device=device, dtype=torch.get_default_dtype()), cmap=plt.get_cmap('gray'))
            #plt.xticks([])
            #plt.yticks([])
            #plt.grid(False)
            #plt.xlabel("comparison, epoch {:s}, loss {:.4f}".format(comp_epoch, loss))


        with torch.no_grad():
            sample = torch.randn(64, 20).to(device=device, dtype=model.get_dtype())
            sample = model.decode(sample).cpu()
            save_image(sample.view(64, 1, 28, 28),
                       'results/sample_' + str(epoch) + '_' + str(model.get_name()) + '.png')
            images.append(sample.view(64,1,28,28))
            captions.append(f"epoch {str(epoch)}")

            #plt.subplot(epochs,2,2*epoch)
            #plt.imshow(sample.view(64, 1, 28, 28).permute(0,2,3,1).reshape(28 * 8, 28 * 8, 1).to(device=device, dtype=torch.get_default_dtype()), cmap=plt.get_cmap('gray'))
            #plt.xticks([])
            #plt.yticks([])
            #plt.grid(False)
            #plt.xlabel("epoch {:f}, test set loss {:.4f}".format(epoch, loss))
    #plt.show()



In [ ]:
test_and_train(model_32, optimizer_32)

In [ ]:
test_and_train(model_w_32, optimizer_w_32)

In [ ]:
test_and_train(model_w_16, optimizer_w_16)

In [ ]:
test_and_train(model_sr_16, optimizer_sr_16)